In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

art_landmark_path = Path("park_feature_join_artwork_landmark.csv")
tree_flower_path = Path("park_feature_join_tree_flower.csv")
park_path = Path("melbourne_city_parks_final.csv")

art_landmark = pd.read_csv(art_landmark_path)
tree_flower = pd.read_csv(tree_flower_path)
park_base = pd.read_csv(park_path)

print("art_landmark shape:", art_landmark.shape)
print("tree_flower shape:", tree_flower.shape)
print("park_base shape:", park_base.shape)

all_features = pd.concat(
    [art_landmark, tree_flower],
    ignore_index=True
)

print("all_features shape:", all_features.shape)
print(all_features["feature_category"].value_counts(dropna=False))
print(all_features["feature_source"].value_counts(dropna=False))

art_landmark shape: (121, 8)
tree_flower shape: (169, 8)
park_base shape: (137, 7)
all_features shape: (290, 8)
feature_category
nature    198
art        89
urban       3
Name: count, dtype: int64
feature_source
tree        148
artwork      86
landmark     35
flower       21
Name: count, dtype: int64


In [7]:
category_counts = (
    all_features
    .groupby(["park_id", "feature_category"])
    .size()
    .reset_index(name="count")
)

category_wide = category_counts.pivot_table(
    index="park_id",
    columns="feature_category",
    values="count",
    fill_value=0
).reset_index()

# 保证三类字段都存在
for col in ["nature", "art", "urban"]:
    if col not in category_wide.columns:
        category_wide[col] = 0

category_wide = category_wide.rename(columns={
    "nature": "nature_count",
    "art": "art_count",
    "urban": "urban_count"
})

category_wide["task_richness_score"] = (
    category_wide["nature_count"]
    + category_wide["art_count"]
    + category_wide["urban_count"]
)

category_wide.head()

feature_category,park_id,art_count,nature_count,urban_count,task_richness_score
0,1,0.0,3.0,0.0,3.0
1,2,1.0,0.0,0.0,1.0
2,3,1.0,2.0,0.0,3.0
3,5,0.0,1.0,0.0,1.0
4,6,0.0,1.0,0.0,1.0


In [8]:
category_cols = ["nature_count", "art_count", "urban_count"]

category_wide["non_zero_category_count"] = (
    category_wide[category_cols] > 0
).sum(axis=1)

category_wide["category_diversity_score"] = (
    category_wide["non_zero_category_count"] / 3
).round(3)

category_wide.head()

feature_category,park_id,art_count,nature_count,urban_count,task_richness_score,non_zero_category_count,category_diversity_score
0,1,0.0,3.0,0.0,3.0,1,0.333
1,2,1.0,0.0,0.0,1.0,1,0.333
2,3,1.0,2.0,0.0,3.0,2,0.667
3,5,0.0,1.0,0.0,1.0,1,0.333
4,6,0.0,1.0,0.0,1.0,1,0.333


In [9]:
park_enriched = park_base.merge(
    category_wide,
    on="park_id",
    how="left"
)

# 没有任何 feature 的 park 填 0
fill_cols = [
    "nature_count",
    "art_count",
    "urban_count",
    "task_richness_score",
    "non_zero_category_count",
    "category_diversity_score"
]

for col in fill_cols:
    park_enriched[col] = park_enriched[col].fillna(0)

park_enriched.head()

,park_id,park_name,ha,latitude,longitude,st_point,description,art_count,nature_count,urban_count,task_richness_score,non_zero_category_count,category_diversity_score
0,1,Alexandra Gardens,5.7493,-37.821402,144.973383,POINT(144.97338326664902 -37.82140175342995),Alexandra Gardens is a public park in Melbourn...,0.0,3.0,0.0,3.0,1.0,0.333
1,2,Alexandra Park,4.8691,-37.824079,144.977433,POINT(144.9774328963564 -37.824079314048966),"Alexandra Park is a public park in Melbourne, ...",1.0,0.0,0.0,1.0,1.0,0.333
2,3,Argyle Square,1.3297,-37.802740,144.965835,POINT(144.96583511130225 -37.802740458199075),"Argyle Square is a public park in Melbourne, M...",1.0,2.0,0.0,3.0,2.0,0.667
3,4,Auckland Lane Reserve,0.0323,-37.777987,144.938940,POINT(144.9389399141761 -37.77798700546881),Auckland Lane Reserve is a public park in Melb...,0.0,0.0,0.0,0.0,0.0,0.000
4,5,Batman Park,1.4328,-37.821785,144.956613,POINT(144.9566130685186 -37.82178529606905),"Batman Park is a public park in Melbourne, Mel...",0.0,1.0,0.0,1.0,1.0,0.333


park_suitability_score =
0.5 * richness_score_norm
+ 0.3 * category_diversity_score
+ 0.2 * area_score

In [11]:
# richness 标准化：10 个或以上 feature 记为满分
park_enriched["richness_score_norm"] = (
    park_enriched["task_richness_score"] / 10
).clip(upper=1)

# area 标准化：10 公顷或以上记为满分
park_enriched["area_score"] = (
    park_enriched["ha"] / 10
).clip(upper=1)

park_enriched["park_suitability_score"] = (
    0.5 * park_enriched["richness_score_norm"]
    + 0.3 * park_enriched["category_diversity_score"]
    + 0.2 * park_enriched["area_score"]
).round(3)

park_enriched[
    [
        "park_id",
        "park_name",
        "ha",
        "nature_count",
        "art_count",
        "urban_count",
        "task_richness_score",
        "category_diversity_score",
        "richness_score_norm",
        "area_score",
        "park_suitability_score"
    ]
].head()

,park_id,park_name,ha,nature_count,art_count,urban_count,task_richness_score,category_diversity_score,richness_score_norm,area_score,park_suitability_score
0,1,Alexandra Gardens,5.7493,3.0,0.0,0.0,3.0,0.333,0.3,0.57493,0.365
1,2,Alexandra Park,4.8691,0.0,1.0,0.0,1.0,0.333,0.1,0.48691,0.247
2,3,Argyle Square,1.3297,2.0,1.0,0.0,3.0,0.667,0.3,0.13297,0.377
3,4,Auckland Lane Reserve,0.0323,0.0,0.0,0.0,0.0,0.000,0.0,0.00323,0.001
4,5,Batman Park,1.4328,1.0,0.0,0.0,1.0,0.333,0.1,0.14328,0.179


In [12]:
def build_feature_summary(row):
    parts = []

    if row["nature_count"] > 0:
        parts.append(f"{int(row['nature_count'])} nature features")
    if row["art_count"] > 0:
        parts.append(f"{int(row['art_count'])} art features")
    if row["urban_count"] > 0:
        parts.append(f"{int(row['urban_count'])} urban discovery features")

    if len(parts) == 0:
        return "No matched task features in this park yet."

    return "This park contains " + ", ".join(parts) + "."

park_enriched["feature_summary"] = park_enriched.apply(
    build_feature_summary,
    axis=1
)

park_enriched[
    [
        "park_name",
        "nature_count",
        "art_count",
        "urban_count",
        "feature_summary"
    ]
].head()

,park_name,nature_count,art_count,urban_count,feature_summary
0,Alexandra Gardens,3.0,0.0,0.0,This park contains 3 nature features.
1,Alexandra Park,0.0,1.0,0.0,This park contains 1 art features.
2,Argyle Square,2.0,1.0,0.0,"This park contains 2 nature features, 1 art fe..."
3,Auckland Lane Reserve,0.0,0.0,0.0,No matched task features in this park yet.
4,Batman Park,1.0,0.0,0.0,This park contains 1 nature features.


In [13]:
top_parks = park_enriched.sort_values(
    "park_suitability_score",
    ascending=False
)

top_parks[
    [
        "park_id",
        "park_name",
        "ha",
        "nature_count",
        "art_count",
        "urban_count",
        "task_richness_score",
        "category_diversity_score",
        "richness_score_norm",
        "area_score",
        "park_suitability_score",
        "feature_summary"
    ]
].head(20)

,park_id,park_name,ha,nature_count,art_count,urban_count,task_richness_score,category_diversity_score,richness_score_norm,area_score,park_suitability_score,feature_summary
35,36,Fitzroy Gardens,26.2848,13.0,18.0,1.0,32.0,1.000,1.0,1.00000,1.000,"This park contains 13 nature features, 18 art ..."
108,109,Shrine of Remembrance Reserve,13.0001,5.0,2.0,1.0,8.0,1.000,0.8,1.00000,0.900,"This park contains 5 nature features, 2 art fe..."
121,122,The Kings Domain Reserve,16.5913,2.0,9.0,0.0,11.0,0.667,1.0,1.00000,0.900,"This park contains 2 nature features, 9 art fe..."
100,101,Royal Park,83.8235,12.0,2.0,0.0,14.0,0.667,1.0,1.00000,0.900,"This park contains 12 nature features, 2 art f..."
14,15,Carlton Gardens South,8.8109,10.0,2.0,0.0,12.0,0.667,1.0,0.88109,0.876,"This park contains 10 nature features, 2 art f..."
7,8,Birrarung Marr,8.1119,4.0,6.0,0.0,10.0,0.667,1.0,0.81119,0.862,"This park contains 4 nature features, 6 art fe..."
36,37,Flagstaff Gardens,7.3521,17.0,4.0,0.0,21.0,0.667,1.0,0.73521,0.847,"This park contains 17 nature features, 4 art f..."
123,124,Treasury Gardens,5.7667,10.0,4.0,0.0,14.0,0.667,1.0,0.57667,0.815,"This park contains 10 nature features, 4 art f..."
34,35,Fawkner Park,41.1885,21.0,0.0,0.0,21.0,0.333,1.0,1.00000,0.800,This park contains 21 nature features.
92,93,Queen Victoria Gardens & Memorial Statue,4.2407,5.0,12.0,0.0,17.0,0.667,1.0,0.42407,0.785,"This park contains 5 nature features, 12 art f..."


In [14]:
from pathlib import Path

final_cols = [
    "park_id",
    "park_name",
    "ha",
    "latitude",
    "longitude",
    "st_point",
    "description",
    "nature_count",
    "art_count",
    "urban_count",
    "task_richness_score",
    "category_diversity_score",
    "richness_score_norm",
    "area_score",
    "park_suitability_score",
    "feature_summary"
]

park_enriched_final = park_enriched[final_cols].copy()

output_path = Path("park_enriched_final.csv")
park_enriched_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Exported:", output_path)
print("Rows:", len(park_enriched_final))
print("Columns:", len(park_enriched_final.columns))

park_enriched_final.head()

Exported: park_enriched_final.csv
Rows: 137
Columns: 16


,park_id,park_name,ha,latitude,longitude,st_point,description,nature_count,art_count,urban_count,task_richness_score,category_diversity_score,richness_score_norm,area_score,park_suitability_score,feature_summary
0,1,Alexandra Gardens,5.7493,-37.821402,144.973383,POINT(144.97338326664902 -37.82140175342995),Alexandra Gardens is a public park in Melbourn...,3.0,0.0,0.0,3.0,0.333,0.3,0.57493,0.365,This park contains 3 nature features.
1,2,Alexandra Park,4.8691,-37.824079,144.977433,POINT(144.9774328963564 -37.824079314048966),"Alexandra Park is a public park in Melbourne, ...",0.0,1.0,0.0,1.0,0.333,0.1,0.48691,0.247,This park contains 1 art features.
2,3,Argyle Square,1.3297,-37.802740,144.965835,POINT(144.96583511130225 -37.802740458199075),"Argyle Square is a public park in Melbourne, M...",2.0,1.0,0.0,3.0,0.667,0.3,0.13297,0.377,"This park contains 2 nature features, 1 art fe..."
3,4,Auckland Lane Reserve,0.0323,-37.777987,144.938940,POINT(144.9389399141761 -37.77798700546881),Auckland Lane Reserve is a public park in Melb...,0.0,0.0,0.0,0.0,0.000,0.0,0.00323,0.001,No matched task features in this park yet.
4,5,Batman Park,1.4328,-37.821785,144.956613,POINT(144.9566130685186 -37.82178529606905),"Batman Park is a public park in Melbourne, Mel...",1.0,0.0,0.0,1.0,0.333,0.1,0.14328,0.179,This park contains 1 nature features.
